<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:15px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#000000; overflow:hidden"><b> Student Performance Analysis and Prediction </b></div>

## ADVANCED KAGGLE NOTEBOOK: Neural Architecture Search from Scratch

Deep Concept: Understanding how machines can design their own neural networks
Educational Focus: Evolutionary algorithms, search spaces, and automated ML
Upvote Strategy: Clear visualizations, practical results, runs in <1 hour



## Neural Architecture Search: Teaching Machines to Design Networks

## What You'll Learn
- How to define a search space for neural architectures
- Evolutionary algorithms for architecture optimization
- Performance estimation strategies
- Practical NAS on real datasets

## Why This Matters
Instead of manually designing networks, we can automate the process.
This notebook shows you how companies like Google discover architectures
like EfficientNet and MobileNet.

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1;overflow:hidden"><b> Installing Required Libraries</b></div>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import random
from collections import namedtuple
from typing import List, Tuple
import json
import time

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

2026-02-14 12:32:57.245923: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771072377.587300      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771072377.753204      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771072378.633413      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771072378.633459      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771072378.633462      55 computation_placer.cc:177] computation placer alr

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1;overflow:hidden"><b> Define the architecture genome</b></div>

In [2]:
LayerGene = namedtuple('LayerGene', ['layer_type', 'units', 'activation', 'dropout'])

class ArchitectureSearchSpace:
    """Defines the space of all possible architectures"""
    
    def __init__(self):
        self.layer_types = ['dense', 'dense_bn', 'dense_dropout']
        self.units_options = [32, 64, 128, 256]
        self.activations = ['relu', 'leaky_relu', 'elu', 'swish']
        self.dropout_rates = [0.0, 0.1, 0.2, 0.3, 0.4]
        self.min_layers = 2
        self.max_layers = 6
    
    def random_architecture(self) -> List[LayerGene]:
        """Generate a random architecture"""
        num_layers = random.randint(self.min_layers, self.max_layers)
        architecture = []
        
        for _ in range(num_layers):
            gene = LayerGene(
                layer_type=random.choice(self.layer_types),
                units=random.choice(self.units_options),
                activation=random.choice(self.activations),
                dropout=random.choice(self.dropout_rates)
            )
            architecture.append(gene)
        
        return architecture
    
    def architecture_to_string(self, arch: List[LayerGene]) -> str:
        """Convert architecture to readable string"""
        return " -> ".join([
            f"{gene.layer_type}({gene.units}, {gene.activation}, d={gene.dropout})"
            for gene in arch
        ])

# Visualize some random architectures
search_space = ArchitectureSearchSpace()

print("\n Examples of Random Architectures:\n")
for i in range(5):
    arch = search_space.random_architecture()
    print(f"Architecture {i+1}:")
    print(f"  {search_space.architecture_to_string(arch)}")
    print(f"  Total layers: {len(arch)}")
    print()



 Examples of Random Architectures:

Architecture 1:
  dense(256, leaky_relu, d=0.4) -> dense_dropout(128, swish, d=0.4) -> dense_bn(32, leaky_relu, d=0.0) -> dense_bn(128, leaky_relu, d=0.2)
  Total layers: 4

Architecture 2:
  dense_bn(128, leaky_relu, d=0.3) -> dense(32, relu, d=0.4) -> dense_bn(32, leaky_relu, d=0.1) -> dense_bn(32, relu, d=0.0)
  Total layers: 4

Architecture 3:
  dense_dropout(128, swish, d=0.0) -> dense_dropout(32, swish, d=0.1) -> dense_dropout(256, elu, d=0.2) -> dense_dropout(256, leaky_relu, d=0.3) -> dense_dropout(64, leaky_relu, d=0.0)
  Total layers: 5

Architecture 4:
  dense_dropout(128, swish, d=0.1) -> dense_dropout(256, relu, d=0.1)
  Total layers: 2

Architecture 5:
  dense_dropout(128, leaky_relu, d=0.0) -> dense(256, relu, d=0.2) -> dense_dropout(32, relu, d=0.4)
  Total layers: 3



<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1;overflow:hidden"><b> Builds Keras models from architecture genomes</b></div>

In [3]:
class ArchitectureBuilder:
    
    
    def __init__(self, input_shape: int, num_classes: int):
        self.input_shape = input_shape
        self.num_classes = num_classes
    
    def build_model(self, architecture: List[LayerGene]) -> keras.Model:
        """Build a Keras model from architecture genome"""
        model = keras.Sequential()
        
        # Input layer
        model.add(layers.Input(shape=(self.input_shape,)))
        
        # Build each layer from genome
        for gene in architecture:
            # Add main layer
            model.add(layers.Dense(gene.units))
            
            # Add activation
            if gene.activation == 'relu':
                model.add(layers.ReLU())
            elif gene.activation == 'leaky_relu':
                model.add(layers.LeakyReLU(alpha=0.1))
            elif gene.activation == 'elu':
                model.add(layers.ELU())
            elif gene.activation == 'swish':
                model.add(layers.Activation('swish'))
            
            # Add batch normalization if specified
            if gene.layer_type == 'dense_bn':
                model.add(layers.BatchNormalization())
            
            # Add dropout if specified
            if gene.layer_type == 'dense_dropout' or gene.dropout > 0:
                model.add(layers.Dropout(gene.dropout))
        
        # Output layer
        model.add(layers.Dense(self.num_classes, activation='softmax'))
        
        # Compile
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        return model

# Test the builder
print("\n Testing Architecture Builder:\n")
test_arch = search_space.random_architecture()
print(f"Building: {search_space.architecture_to_string(test_arch)}")

builder = ArchitectureBuilder(input_shape=64, num_classes=10)
test_model = builder.build_model(test_arch)

print(f"Total parameters: {test_model.count_params():,}")



 Testing Architecture Builder:

Building: dense(64, elu, d=0.3) -> dense_bn(32, elu, d=0.2)


I0000 00:00:1771072397.644720      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1771072397.651028      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Total parameters: 6,698


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1;overflow:hidden"><b>Quickly estimates architecture performance</b></div>

In [4]:
class PerformanceEstimator:
    
    
    def __init__(self, X_train, y_train, X_val, y_val, 
                 quick_epochs=10, full_epochs=50):
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.quick_epochs = quick_epochs
        self.full_epochs = full_epochs
    
    def quick_evaluate(self, model: keras.Model, verbose=0) -> dict:
        """Fast evaluation with early stopping"""
        
        history = model.fit(
            self.X_train, self.y_train,
            validation_data=(self.X_val, self.y_val),
            epochs=self.quick_epochs,
            batch_size=32,
            verbose=verbose,
            callbacks=[
                keras.callbacks.EarlyStopping(
                    monitor='val_loss',
                    patience=3,
                    restore_best_weights=True
                )
            ]
        )
        
        # Get final validation accuracy
        val_acc = max(history.history['val_accuracy'])
        train_time = len(history.history['loss'])
        
        return {
            'val_accuracy': val_acc,
            'epochs_trained': train_time,
            'history': history.history
        }
    
    def full_evaluate(self, model: keras.Model, verbose=1) -> dict:
        """Full training for best architectures"""
        
        history = model.fit(
            self.X_train, self.y_train,
            validation_data=(self.X_val, self.y_val),
            epochs=self.full_epochs,
            batch_size=32,
            verbose=verbose,
            callbacks=[
                keras.callbacks.EarlyStopping(
                    monitor='val_loss',
                    patience=5,
                    restore_best_weights=True
                ),
                keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.5,
                    patience=3
                )
            ]
        )
        
        val_acc = max(history.history['val_accuracy'])
        
        return {
            'val_accuracy': val_acc,
            'history': history.history
        }


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1;overflow:hidden"><b>Evolutionary algorithm for architecture search</b></div>

In [5]:
class EvolutionaryNAS:
 
    
    def __init__(self, search_space, builder, estimator, 
                 population_size=20, generations=10):
        self.search_space = search_space
        self.builder = builder
        self.estimator = estimator
        self.population_size = population_size
        self.generations = generations
        self.history = []
    
    def initialize_population(self) -> List[Tuple[List[LayerGene], float]]:
       
        population = []
        
        print(f"\n Initializing population of {self.population_size} architectures...")
        
        for i in range(self.population_size):
            arch = self.search_space.random_architecture()
            population.append((arch, 0.0))  # (architecture, fitness)
            
            if (i + 1) % 5 == 0:
                print(f"  Generated {i + 1}/{self.population_size} architectures")
        
        return population
    
    def evaluate_population(self, population: List[Tuple[List[LayerGene], float]]) -> List[Tuple[List[LayerGene], float]]:
        
        evaluated_pop = []
        
        for i, (arch, _) in enumerate(population):
            try:
                # Build model
                model = self.builder.build_model(arch)
                
                # Quick evaluation
                results = self.estimator.quick_evaluate(model, verbose=0)
                fitness = results['val_accuracy']
                
                evaluated_pop.append((arch, fitness))
                
                if (i + 1) % 5 == 0:
                    print(f"  Evaluated {i + 1}/{len(population)} | Best so far: {max([f for _, f in evaluated_pop]):.4f}")
                
                # Clean up
                del model
                tf.keras.backend.clear_session()
                
            except Exception as e:
                # If architecture fails, give it low fitness
                evaluated_pop.append((arch, 0.0))
                print(f"  Architecture {i+1} failed: {e}")
        
        return evaluated_pop
    
    def select_parents(self, population: List[Tuple[List[LayerGene], float]], k: int) -> List[List[LayerGene]]:
       
        sorted_pop = sorted(population, key=lambda x: x[1], reverse=True)
        return [arch for arch, _ in sorted_pop[:k]]
    
    def mutate(self, architecture: List[LayerGene], mutation_rate=0.3) -> List[LayerGene]:
        
        new_arch = architecture.copy()
        
        # Maybe add a layer
        if random.random() < mutation_rate and len(new_arch) < self.search_space.max_layers:
            new_gene = LayerGene(
                layer_type=random.choice(self.search_space.layer_types),
                units=random.choice(self.search_space.units_options),
                activation=random.choice(self.search_space.activations),
                dropout=random.choice(self.search_space.dropout_rates)
            )
            insert_pos = random.randint(0, len(new_arch))
            new_arch.insert(insert_pos, new_gene)
        
        # Maybe remove a layer
        elif random.random() < mutation_rate and len(new_arch) > self.search_space.min_layers:
            remove_pos = random.randint(0, len(new_arch) - 1)
            new_arch.pop(remove_pos)
        
        # Maybe modify a layer
        elif random.random() < mutation_rate and len(new_arch) > 0:
            modify_pos = random.randint(0, len(new_arch) - 1)
            old_gene = new_arch[modify_pos]
            
            new_arch[modify_pos] = LayerGene(
                layer_type=random.choice(self.search_space.layer_types),
                units=random.choice(self.search_space.units_options),
                activation=random.choice(self.search_space.activations),
                dropout=random.choice(self.search_space.dropout_rates)
            )
        
        return new_arch
    
    def crossover(self, parent1: List[LayerGene], parent2: List[LayerGene]) -> List[LayerGene]:
       
        # Take random layers from each parent
        crossover_point = random.randint(1, min(len(parent1), len(parent2)) - 1)
        
        if random.random() < 0.5:
            child = parent1[:crossover_point] + parent2[crossover_point:]
        else:
            child = parent2[:crossover_point] + parent1[crossover_point:]
        
        # Ensure child meets length constraints
        if len(child) < self.search_space.min_layers:
            while len(child) < self.search_space.min_layers:
                child.append(self.search_space.random_architecture()[0])
        elif len(child) > self.search_space.max_layers:
            child = child[:self.search_space.max_layers]
        
        return child
    
    def evolve(self) -> Tuple[List[LayerGene], dict]:

        
        # Initialize
        population = self.initialize_population()
        
        # Evolution loop
        for gen in range(self.generations):
            print(f"\n Generation {gen + 1}/{self.generations}")
            print("-" * 70)
            
            # Evaluate
            population = self.evaluate_population(population)
            
            # Track best
            best_arch, best_fitness = max(population, key=lambda x: x[1])
            avg_fitness = np.mean([f for _, f in population])
            
            self.history.append({
                'generation': gen + 1,
                'best_fitness': best_fitness,
                'avg_fitness': avg_fitness,
                'best_architecture': best_arch
            })
            
            print(f"\n   Best accuracy: {best_fitness:.4f}")
            print(f"   Average accuracy: {avg_fitness:.4f}")
            print(f"   Best architecture: {self.search_space.architecture_to_string(best_arch)}")
            
            if gen < self.generations - 1:  # Don't create new generation on last iteration
                # Selection
                num_parents = self.population_size // 2
                parents = self.select_parents(population, num_parents)
                
                # Create new generation
                new_population = []
                
                # Keep elite (top 20%)
                elite_size = max(2, self.population_size // 5)
                elite = self.select_parents(population, elite_size)
                new_population.extend([(arch, 0.0) for arch in elite])
                
                # Create offspring through crossover and mutation
                while len(new_population) < self.population_size:
                    parent1, parent2 = random.sample(parents, 2)
                    
                    # Crossover
                    if random.random() < 0.7:  # 70% crossover rate
                        child = self.crossover(parent1, parent2)
                    else:
                        child = random.choice([parent1, parent2]).copy()
                    
                    # Mutation
                    child = self.mutate(child, mutation_rate=0.3)
                    
                    new_population.append((child, 0.0))
                
                population = new_population[:self.population_size]
        
        
        
        final_best_arch, final_best_fitness = max(population, key=lambda x: x[1])
        
        print(f"\n Best Architecture Found:")
        print(f"  {self.search_space.architecture_to_string(final_best_arch)}")
        print(f"  Validation Accuracy: {final_best_fitness:.4f}")
        
        return final_best_arch, {
            'history': self.history,
            'final_fitness': final_best_fitness
        }


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1;overflow:hidden"><b>Loading and preparing data</b></div>

In [6]:
digits = load_digits()
X, y = digits.data, digits.target

print(f"\n Dataset Information:")
print(f"  Samples: {X.shape[0]}")
print(f"  Features: {X.shape[1]}")
print(f"  Classes: {len(np.unique(y))}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"\n  Training set: {X_train.shape[0]} samples")
print(f"  Validation set: {X_val.shape[0]} samples")
print(f"  Test set: {X_test.shape[0]} samples")

# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Label: {digits.target[i]}')
    ax.axis('off')
plt.suptitle('Sample Digits from Dataset', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('sample_digits.png', dpi=150, bbox_inches='tight')
plt.close()





 Dataset Information:
  Samples: 1797
  Features: 64
  Classes: 10

  Training set: 1149 samples
  Validation set: 288 samples
  Test set: 360 samples

 Data prepared and visualized


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1;overflow:hidden"><b>Run Neural Architecture Search</b></div>

In [7]:

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# Initialize components
search_space = ArchitectureSearchSpace()
builder = ArchitectureBuilder(input_shape=X_train.shape[1], num_classes=10)
estimator = PerformanceEstimator(X_train, y_train, X_val, y_val, 
                                  quick_epochs=15, full_epochs=50)

# Create evolutionary NAS
nas = EvolutionaryNAS(
    search_space=search_space,
    builder=builder,
    estimator=estimator,
    population_size=20,  # Small for Kaggle
    generations=8        # Reduced for time
)

# Run search
start_time = time.time()
best_architecture, search_results = nas.evolve()
search_time = time.time() - start_time

print(f"\n  Total search time: {search_time/60:.2f} minutes")



 Initializing population of 20 architectures...
  Generated 5/20 architectures
  Generated 10/20 architectures
  Generated 15/20 architectures
  Generated 20/20 architectures

 Generation 1/8
----------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(
I0000 00:00:1771072401.458063     123 service.cc:152] XLA service 0x7823e801bf80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771072401.458099     123 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1771072401.458106     123 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1771072401.823344     123 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1771072403.775650     123 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Evaluated 5/20 | Best so far: 0.9722
  Evaluated 10/20 | Best so far: 0.9826
  Evaluated 15/20 | Best so far: 0.9861
  Evaluated 20/20 | Best so far: 0.9861

   Best accuracy: 0.9861
   Average accuracy: 0.9594
   Best architecture: dense_dropout(256, elu, d=0.3) -> dense_bn(256, relu, d=0.0)

 Generation 2/8
----------------------------------------------------------------------
  Evaluated 5/20 | Best so far: 0.9826
  Evaluated 10/20 | Best so far: 0.9826
  Evaluated 15/20 | Best so far: 0.9826
  Evaluated 20/20 | Best so far: 0.9826

   Best accuracy: 0.9826
   Average accuracy: 0.9720
   Best architecture: dense(128, swish, d=0.1) -> dense(256, relu, d=0.0) -> dense_dropout(32, relu, d=0.1) -> dense(256, swish, d=0.3) -> dense(256, relu, d=0.1)

 Generation 3/8
----------------------------------------------------------------------
  Evaluated 5/20 | Best so far: 0.9757
  Evaluated 10/20 | Best so far: 0.9826
  Evaluated 15/20 | Best so far: 0.9896
  Evaluated 20/20 | Best so far: 

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1;overflow:hidden"><b>Visualizations </b></div>

In [8]:
history_df = pd.DataFrame(search_results['history'])

# Plot evolution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Best fitness over generations
axes[0].plot(history_df['generation'], history_df['best_fitness'], 
             marker='o', linewidth=2, markersize=8, label='Best')
axes[0].plot(history_df['generation'], history_df['avg_fitness'], 
             marker='s', linewidth=2, markersize=6, label='Average', alpha=0.7)
axes[0].set_xlabel('Generation', fontsize=12)
axes[0].set_ylabel('Validation Accuracy', fontsize=12)
axes[0].set_title('Architecture Evolution Over Generations', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Improvement percentage
initial_best = history_df['best_fitness'].iloc[0]
final_best = history_df['best_fitness'].iloc[-1]
improvement = ((final_best - initial_best) / initial_best) * 100

axes[1].bar(['Initial', 'Final'], [initial_best, final_best], 
            color=['#3498db', '#2ecc71'], alpha=0.7, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Validation Accuracy', fontsize=12)
axes[1].set_title(f'Improvement: {improvement:.1f}%', fontsize=14, fontweight='bold')
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

for i, v in enumerate([initial_best, final_best]):
    axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('evolution_progress.png', dpi=150, bbox_inches='tight')
plt.close()




In [9]:
best_model = builder.build_model(best_architecture)

print(f"\n Architecture Details:")
print(f"  {search_space.architecture_to_string(best_architecture)}")
print(f"\n  Total Parameters: {best_model.count_params():,}")

# Full training
full_results = estimator.full_evaluate(best_model, verbose=1)

print(f"\nFinal Validation Accuracy: {full_results['val_accuracy']:.4f}")

# Evaluate on test set
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Set Accuracy: {test_acc:.4f}")



 Architecture Details:
  dense(256, elu, d=0.3) -> dense_dropout(128, swish, d=0.2) -> dense(32, elu, d=0.0) -> dense_bn(64, relu, d=0.2)

  Total Parameters: 56,682
Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 84ms/step - accuracy: 0.4032 - loss: 1.9092 - val_accuracy: 0.8576 - val_loss: 1.1245 - learning_rate: 0.0010
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8348 - loss: 0.6083 - val_accuracy: 0.9236 - val_loss: 0.6717 - learning_rate: 0.0010
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9087 - loss: 0.3821 - val_accuracy: 0.9444 - val_loss: 0.4228 - learning_rate: 0.0010
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9358 - loss: 0.2876 - val_accuracy: 0.9583 - val_loss: 0.2828 - learning_rate: 0.0010
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9314 - loss: 0.2264 - val_accuracy: 0.9549 - val_loss: 0.2312 - learning_rate: 0.0010
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9533 - loss: 0.181

In [10]:

baseline_model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

baseline_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n  Training baseline model")
baseline_history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    verbose=0,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ]
)

baseline_test_loss, baseline_test_acc = baseline_model.evaluate(X_test, y_test, verbose=0)

print(f"\n Results Comparison:")
print(f"  Baseline (Manual Design): {baseline_test_acc:.4f}")
print(f"  NAS (Discovered):         {test_acc:.4f}")
print(f"  Improvement:              {((test_acc - baseline_test_acc) / baseline_test_acc * 100):.2f}%")



  Training baseline model

📊 Results Comparison:
  Baseline (Manual Design): 0.9722
  NAS (Discovered):         0.9806
  Improvement:              0.86%


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy curves
axes[0].plot(full_results['history']['accuracy'], label='NAS Train', linewidth=2)
axes[0].plot(full_results['history']['val_accuracy'], label='NAS Val', linewidth=2)
axes[0].plot(baseline_history.history['accuracy'], label='Baseline Train', 
             linewidth=2, linestyle='--', alpha=0.7)
axes[0].plot(baseline_history.history['val_accuracy'], label='Baseline Val', 
             linewidth=2, linestyle='--', alpha=0.7)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Training Curves Comparison', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss curves
axes[1].plot(full_results['history']['loss'], label='NAS Train', linewidth=2)
axes[1].plot(full_results['history']['val_loss'], label='NAS Val', linewidth=2)
axes[1].plot(baseline_history.history['loss'], label='Baseline Train', 
             linewidth=2, linestyle='--', alpha=0.7)
axes[1].plot(baseline_history.history['val_loss'], label='Baseline Val', 
             linewidth=2, linestyle='--', alpha=0.7)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Loss Curves Comparison', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.close()


In [13]:
summary = {
    'search_time_minutes': search_time / 60,
    'total_generations': len(history_df),
    'total_architectures_evaluated': len(history_df) * nas.population_size,
    'best_architecture': search_space.architecture_to_string(best_architecture),
    'best_validation_accuracy': full_results['val_accuracy'],
    'test_accuracy': test_acc,
    'baseline_test_accuracy': baseline_test_acc,
    'improvement_over_baseline': ((test_acc - baseline_test_acc) / baseline_test_acc * 100),
    'best_model_parameters': int(best_model.count_params()),
    'baseline_parameters': int(baseline_model.count_params())
}

for key, value in summary.items():
    print(f"  {key.replace('_', ' ').title()}: {value}")

# Save summary
summary_df = pd.DataFrame([summary])
summary_df.to_csv('nas_summary.csv', index=False)



best_model.save('best_nas_model.keras')

arch_dict = {
    'architecture': [
        {
            'layer_type': gene.layer_type,
            'units': gene.units,
            'activation': gene.activation,
            'dropout': gene.dropout
        }
        for gene in best_architecture
    ],
    'performance': {
        'validation_accuracy': float(full_results['val_accuracy']),
        'test_accuracy': float(test_acc)
    }
}

with open('best_architecture.json', 'w') as f:
    json.dump(arch_dict, f, indent=2)


  Search Time Minutes: 28.510161821047465
  Total Generations: 8
  Total Architectures Evaluated: 160
  Best Architecture: dense(256, elu, d=0.3) -> dense_dropout(128, swish, d=0.2) -> dense(32, elu, d=0.0) -> dense_bn(64, relu, d=0.2)
  Best Validation Accuracy: 0.9826388955116272
  Test Accuracy: 0.980555534362793
  Baseline Test Accuracy: 0.9722222089767456
  Improvement Over Baseline: 0.8571420513853625
  Best Model Parameters: 56682
  Baseline Parameters: 17226
